# TEBD Helper APIs in Operator Space

This notebook mirrors the physical-spin helper walkthrough, but now in operator space.

The same helper stack is used:

1. `local_gates_from_hamiltonians`
2. `tebd_evolution_from_hamiltonians`
3. `tebd_strang_schedule`
4. `tebd_strang_evolution`

The main extra ingredient is `map_hamiltonian=pauli_gate_from_hamiltonian`, which converts each dense physical Hamiltonian into the corresponding Pauli-basis superoperator gate.


In [ ]:
using LinearAlgebra
using Plots
using ITensors
using ITensorMPS
using MPSToolkit

default(size=(960, 680), linewidth=2, markersize=4, legend=:best)

spins = spinhalf_matrices()

function tfim_bond_hamiltonian(; J::Real=1.0, g::Real=0.8)
    return J * kron(spins.Sz, spins.Sz) +
           0.5 * g * (kron(spins.Sx, spins.I) + kron(spins.I, spins.Sx))
end

function local_pauli_z_profile(state::MPS)
    sites = siteinds(state)
    nsites = length(state)
    return [
        real(inner(pauli_basis_state(sites, [site == j ? "Z" : "I" for site in 1:nsites]), state))
        for j in 1:nsites
    ]
end


## 1. `local_gates_from_hamiltonians` with `map_hamiltonian`

For operator-space TEBD, the local Hamiltonians are still ordinary dense physical Hamiltonians.
The helper stays the same, but we pass a mapping function telling it how to turn those Hamiltonians into Pauli-basis superoperators.


In [ ]:
dt = 0.05
hbond = tfim_bond_hamiltonian()

single_gate = local_gates_from_hamiltonians(hbond, dt; map_hamiltonian=pauli_gate_from_hamiltonian)
vector_gates = local_gates_from_hamiltonians([hbond, 0.7 .* hbond], dt; map_hamiltonian=pauli_gate_from_hamiltonian)
callable_gates = local_gates_from_hamiltonians((bond, index) -> (0.3 * index) .* hbond, dt; map_hamiltonian=pauli_gate_from_hamiltonian)

println("single operator-space gate size  = ", size(single_gate))
println("vector gate sizes                = ", [size(g) for g in vector_gates])
println("callable operator gate (1, 1)    = ", size(callable_gates(1, 1)))


## 2. `tebd_evolution_from_hamiltonians`

We can build a scheduled operator-space evolution exactly the same way, just with the extra mapping hook.


In [ ]:
nsites = 6
sites = pauli_siteinds(nsites)
initial = pauli_basis_state(sites, ["Z", "I", "I", "I", "I", "I"])
state_manual = copy(initial)
state_helper = copy(initial)

schedule = [1, 2, 3, 4, 5, 5, 4, 3, 2, 1]
manual_gates = fill(local_gates_from_hamiltonians(hbond, dt; map_hamiltonian=pauli_gate_from_hamiltonian), length(schedule))
manual_evolution = LocalGateEvolution(manual_gates, dt; schedule=schedule, maxdim=64, cutoff=1e-10)
helper_evolution = tebd_evolution_from_hamiltonians(fill(hbond, length(schedule)), dt; map_hamiltonian=pauli_gate_from_hamiltonian, schedule=schedule, maxdim=64, cutoff=1e-10)

evolve!(state_manual, manual_evolution)
evolve!(state_helper, helper_evolution)

println("manual/helper operator overlap = ", abs(inner(state_manual, state_helper)))
bar(1:nsites, local_pauli_z_profile(state_helper); xlabel="site", ylabel="local Z amplitude", title="Helper-built operator evolution", label="helper evolution")


## 3. `tebd_strang_schedule`

The scheduling helper itself is unchanged in operator space. The same odd-even-odd schedule is used; only the Hamiltonian-to-gate map changes.


In [ ]:
schedule_strang, weights_strang = tebd_strang_schedule(nsites)
println("Strang schedule = ", schedule_strang)
println("Strang weights  = ", weights_strang)
plot(1:length(weights_strang), weights_strang; marker=:circle, xlabel="schedule index", ylabel="weight", title="Strang weights reused in operator space", label="weight")


## 4. `tebd_strang_evolution`

This is the most compact operator-space route: provide a local Hamiltonian builder and the Pauli-basis map, then call `evolve!`.


In [ ]:
state_strang = copy(initial)
strang_evolution = tebd_strang_evolution(
    nsites,
    dt;
    local_hamiltonian=(bond, weight) -> weight * tfim_bond_hamiltonian(J=1.0, g=0.6),
    map_hamiltonian=pauli_gate_from_hamiltonian,
    maxdim=64,
    cutoff=1e-10,
)

profiles = [local_pauli_z_profile(state_strang)]
for _ in 1:8
    evolve!(state_strang, strang_evolution)
    push!(profiles, local_pauli_z_profile(state_strang))
end

heatmap(0:8, 1:nsites, reduce(hcat, profiles); xlabel="Strang step", ylabel="site", title="Packed operator-space Strang helper workflow", colorbar_title="local Z amplitude")


## 5. Manual vs helper route

The packed helper route and the more explicit route should agree when they encode the same schedule and the same local Hamiltonian data.


In [ ]:
state_a = copy(initial)
state_b = copy(initial)

schedule_check, weights_check = tebd_strang_schedule(nsites)
hamiltonians = [weight * tfim_bond_hamiltonian(J=1.0, g=0.6) for weight in weights_check]
manual = tebd_evolution_from_hamiltonians(hamiltonians, dt; map_hamiltonian=pauli_gate_from_hamiltonian, schedule=schedule_check, maxdim=64, cutoff=1e-10)
packed = tebd_strang_evolution(nsites, dt; local_hamiltonian=(bond, weight) -> weight * tfim_bond_hamiltonian(J=1.0, g=0.6), map_hamiltonian=pauli_gate_from_hamiltonian, maxdim=64, cutoff=1e-10)

evolve!(state_a, manual)
evolve!(state_b, packed)
println("manual vs packed operator overlap = ", abs(inner(state_a, state_b)))
